In [3]:
from pathlib import Path
import numpy as np

# Find the repo root, even if the notebook is in a subfolder.
repo = Path.cwd()
while not (repo / "data/ready_full/fold0").exists():
    if repo == repo.parent:
        raise FileNotFoundError("Could not find data/ready_full/fold0")
    repo = repo.parent

for split in ("train", "val"):
    mask = np.load(repo / f"data/ready_full/fold0/mask_{split}.npy")

    # Left padding means each row can only go from 0 to 1, never 1 to 0.
    binary = np.all((mask == 0) | (mask == 1))
    left_padded = np.all(np.diff(mask, axis=1) >= 0, axis=1)
    has_real_steps = np.any(mask == 1, axis=1)

    print(f"{split}: shape={mask.shape}")
    print(f"  only 0/1 values: {binary}")
    print(f"  valid left-padded rows: {(left_padded & has_real_steps).sum()}/{len(mask)}")

    # Show one sequence that actually has padding.
    i = np.flatnonzero(mask[:, 0] == 0)[0]
    first_real = np.flatnonzero(mask[i] == 1)[0]
    print(f"  example row {i}: {first_real} zeros, {mask.shape[1] - first_real} ones")
    print(f"  around boundary: {mask[i, first_real-3:first_real+3].astype(int).tolist()}")

train: shape=(6427, 224)
  only 0/1 values: True
  valid left-padded rows: 6427/6427
  example row 0: 167 zeros, 57 ones
  around boundary: [0, 0, 0, 1, 1, 1]
val: shape=(1627, 224)
  only 0/1 values: True
  valid left-padded rows: 1627/1627
  example row 0: 156 zeros, 68 ones
  around boundary: [0, 0, 0, 1, 1, 1]


In [33]:
# Fold 0 timing run: setup and hyperparameters
from pathlib import Path
import copy
import random
import time

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

repo = Path.cwd()
while not (repo / "data/ready_full/fold0").is_dir():
    if repo == repo.parent:
        raise FileNotFoundError("Could not find data/ready_full/fold0")
    repo = repo.parent

import sys
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

from model.baseline import FullSensorCNNGRU
from utils.evaluation import competition_score

FOLD = 0
SEED = 42
BATCH_SIZE = 16
MAX_EPOCHS = 50
PATIENCE = 10
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 1e-4
MIN_LEARNING_RATE = 1e-5
GRAD_CLIP_NORM = 1.0
NUM_WORKERS = 0  # Simple and reliable for Windows notebooks.

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable; this timing run is intended for your GPU.")
device = torch.device("cuda")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
print(f"Fold {FOLD}, seed {SEED}, device: {torch.cuda.get_device_name(0)}")


Fold 0, seed 42, device: NVIDIA GeForce RTX 3050 Laptop GPU


In [34]:
# Load only the selected fold. Memory mapping avoids copying the ~2 GB training array into RAM.
class GestureDataset(Dataset):
    def __init__(self, fold_dir, split):
        self.x = np.load(fold_dir / f"X_{split}.npy", mmap_mode="r")
        self.mask = np.load(fold_dir / f"mask_{split}.npy", mmap_mode="r")
        self.y = np.load(fold_dir / f"y_gesture_{split}.npy", mmap_mode="r")
        if not (len(self.x) == len(self.mask) == len(self.y)):
            raise ValueError(f"Mismatched {split} array lengths")

    def __len__(self):
        return len(self.y)

    def __getitem__(self, index):
        # Make writable per-sample copies before converting read-only memmap slices to tensors.
        x = torch.from_numpy(np.array(self.x[index], dtype=np.float32, copy=True))
        mask = torch.from_numpy(np.array(self.mask[index], copy=True))
        y = torch.tensor(int(self.y[index]), dtype=torch.long)
        return x, mask, y

fold_dir = repo / "data/ready_full" / f"fold{FOLD}"
train_dataset = GestureDataset(fold_dir, "train")
val_dataset = GestureDataset(fold_dir, "val")
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True,
    generator=torch.Generator().manual_seed(SEED),
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)
print(f"Train: {len(train_dataset)} sequences; validation: {len(val_dataset)} sequences")


Train: 6427 sequences; validation: 1627 sequences


In [18]:
# Single 18-class baseline head; the evaluation function collapses non-target classes.
model = FullSensorCNNGRU(num_classes=18).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=MAX_EPOCHS, eta_min=MIN_LEARNING_RATE,
)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


Trainable parameters: 524,946


In [35]:
from utils.train import train_one_epoch, validate


# Training duration estimate

Run training on fold0 only, to estimate total time needed for cross validation across all folds

In [8]:
# Run ONE fold. Stop when the competition score has not improved for PATIENCE epochs.
history = []
best_score = float("-inf")
best_epoch = 0
best_weights = None
epochs_without_improvement = 0

torch.cuda.synchronize()
run_start = time.perf_counter()
for epoch in range(1, MAX_EPOCHS + 1):
    epoch_start = time.perf_counter()
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device, GRAD_CLIP_NORM)
    val_loss, metrics, _, _ = validate(model, val_loader, criterion, device)
    scheduler.step()
    torch.cuda.synchronize()
    epoch_seconds = time.perf_counter() - epoch_start

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        **metrics,
        "seconds": epoch_seconds,
    })
    print(
        f"Epoch {epoch:02d} | train loss {train_loss:.4f} | val loss {val_loss:.4f} "
        f"| binary F1 {metrics['binary_f1']:.4f} | macro F1 {metrics['macro_f1']:.4f} "
        f"| score {metrics['score']:.4f} | {epoch_seconds:.1f}s"
    )

    if metrics["score"] > best_score:
        best_score = metrics["score"]
        best_epoch = epoch
        best_weights = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print(f"Early stopping after {PATIENCE} epochs without improvement.")
            break

fold_seconds = time.perf_counter() - run_start
model.load_state_dict(best_weights)
print(f"Best epoch: {best_epoch}; best validation score: {best_score:.4f}")
print(f"Fold {FOLD} training + validation time: {fold_seconds / 60:.1f} minutes")


Epoch 01 | train loss 1.7105 | val loss 1.3784 | binary F1 0.9443 | macro F1 0.4516 | score 0.6979 | 35.0s
Epoch 02 | train loss 1.1426 | val loss 1.1111 | binary F1 0.9702 | macro F1 0.5660 | score 0.7681 | 22.6s
Epoch 03 | train loss 0.9311 | val loss 0.9183 | binary F1 0.9716 | macro F1 0.6069 | score 0.7892 | 15.2s
Epoch 04 | train loss 0.7863 | val loss 0.8925 | binary F1 0.9738 | macro F1 0.6115 | score 0.7927 | 16.1s
Epoch 05 | train loss 0.6948 | val loss 0.8467 | binary F1 0.9829 | macro F1 0.6320 | score 0.8075 | 16.7s
Epoch 06 | train loss 0.6183 | val loss 0.9187 | binary F1 0.9778 | macro F1 0.5964 | score 0.7871 | 16.0s
Epoch 07 | train loss 0.5452 | val loss 0.9444 | binary F1 0.9770 | macro F1 0.6144 | score 0.7957 | 15.1s
Epoch 08 | train loss 0.4870 | val loss 0.9761 | binary F1 0.9799 | macro F1 0.6205 | score 0.8002 | 13.3s
Epoch 09 | train loss 0.4373 | val loss 0.9572 | binary F1 0.9830 | macro F1 0.6263 | score 0.8046 | 13.9s
Epoch 10 | train loss 0.3776 | val lo

In [9]:
# Rough projection. Fold sizes and early-stopping epochs can differ.
print(f"One fold, one seed: {fold_seconds / 60:.1f} minutes")
print(f"Five folds, one seed: about {5 * fold_seconds / 60:.1f} minutes")
print(f"Five folds, 2 seeds: about {10 * fold_seconds / 60:.1f} minutes")
print(f"Five folds, 3 seeds: about {15 * fold_seconds / 60:.1f} minutes")


One fold, one seed: 9.2 minutes
Five folds, one seed: about 46.1 minutes
Five folds, 2 seeds: about 92.2 minutes
Five folds, 3 seeds: about 138.3 minutes


# Five-fold baseline cross-validation (one seed)

Run the setup, dataset, and training-function cells above first. If starting a fresh kernel, skip the earlier **Run ONE fold** and duration-estimate cells; this section trains fold 0 again from scratch. Each fold gets a fresh model, optimizer, scheduler, and participant-separated training/validation split. No cells in this section have been executed.

In [20]:
# Train the same baseline independently on folds 0-4, using one fixed seed.
from tqdm.auto import tqdm

checkpoint_dir = repo / "model"
checkpoint_dir.mkdir(parents=True, exist_ok=True)
model_kwargs = {"num_classes": 18, "gru_hidden_size": 128, "dropout": 0.2}

cv_results = []
cv_history = []
oof_true_parts, oof_pred_parts, oof_fold_parts = [], [], []

for fold in range(5):
    # Reset the seed so each fold starts reproducibly from a fresh model.
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    fold_dir = repo / "data/ready_full" / f"fold{fold}"
    fold_train = GestureDataset(fold_dir, "train")
    fold_val = GestureDataset(fold_dir, "val")
    fold_train_loader = DataLoader(
        fold_train, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=True,
        generator=torch.Generator().manual_seed(SEED),
    )
    fold_val_loader = DataLoader(
        fold_val, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True,
    )

    fold_model = FullSensorCNNGRU(**model_kwargs).to(device)
    checkpoint_path = checkpoint_dir / f"baseline_fold{fold}_seed{SEED}.pt"
    fold_criterion = nn.CrossEntropyLoss()
    fold_optimizer = torch.optim.AdamW(
        fold_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
    )
    fold_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        fold_optimizer, T_max=MAX_EPOCHS, eta_min=MIN_LEARNING_RATE,
    )

    best_score = float("-inf")
    best_epoch = 0
    best_metrics = None
    best_true = best_pred = None
    stale_epochs = 0
    torch.cuda.synchronize()
    fold_start = time.perf_counter()

    epoch_bar = tqdm(range(1, MAX_EPOCHS + 1), desc=f"Fold {fold}", unit="epoch")
    for epoch in epoch_bar:
        epoch_start = time.perf_counter()
        train_loss = train_one_epoch(
            fold_model, fold_train_loader, fold_optimizer, fold_criterion,
            device, GRAD_CLIP_NORM,
        )
        val_loss, metrics, y_true, y_pred = validate(
            fold_model, fold_val_loader, fold_criterion, device,
        )
        fold_scheduler.step()
        torch.cuda.synchronize()
        epoch_seconds = time.perf_counter() - epoch_start

        cv_history.append({
            "fold": fold, "epoch": epoch, "train_loss": train_loss,
            "val_loss": val_loss, **metrics, "seconds": epoch_seconds,
        })
        if metrics["score"] > best_score:
            best_score = metrics["score"]
            best_epoch = epoch
            best_metrics = metrics.copy()
            best_true, best_pred = y_true.copy(), y_pred.copy()
            stale_epochs = 0
            checkpoint = {
                "model_class": "model.baseline.FullSensorCNNGRU",
                "model_kwargs": model_kwargs.copy(),
                "model_state_dict": {
                    name: tensor.detach().cpu().clone()
                    for name, tensor in fold_model.state_dict().items()
                },
                "fold": fold,
                "seed": SEED,
                "best_epoch": epoch,
                "val_metrics": {name: float(value) for name, value in metrics.items()},
                "preprocessing_stats": f"data/ready_full/stats_fold{fold}.npz",
            }
            temporary_path = checkpoint_path.with_suffix(".tmp")
            torch.save(checkpoint, temporary_path)
            temporary_path.replace(checkpoint_path)
        else:
            stale_epochs += 1

        epoch_bar.set_postfix(
            score=f"{metrics['score']:.4f}",
            best=f"{best_score:.4f}",
            seconds=f"{epoch_seconds:.0f}",
        )
        if stale_epochs >= PATIENCE:
            break

    torch.cuda.synchronize()
    fold_seconds = time.perf_counter() - fold_start
    cv_results.append({
        "fold": fold, "train_n": len(fold_train), "val_n": len(fold_val),
        "best_epoch": best_epoch, **best_metrics, "minutes": fold_seconds / 60,
        "checkpoint": str(checkpoint_path),
    })
    oof_true_parts.append(best_true)
    oof_pred_parts.append(best_pred)
    oof_fold_parts.append(np.full(len(best_true), fold, dtype=np.int8))
    print(
        f"Fold {fold}: best epoch {best_epoch}, "
        f"binary F1 {best_metrics['binary_f1']:.4f}, "
        f"macro F1 {best_metrics['macro_f1']:.4f}, "
        f"score {best_score:.4f}, time {fold_seconds / 60:.1f} min, "
        f"saved {checkpoint_path.name}"
    )

oof_true = np.concatenate(oof_true_parts)
oof_pred = np.concatenate(oof_pred_parts)
oof_fold = np.concatenate(oof_fold_parts)


Fold 0:  58%|█████▊    | 29/50 [06:38<04:48, 13.75s/epoch, best=0.8184, score=0.8110, seconds=13]


Fold 0: best epoch 20, binary F1 0.9853, macro F1 0.6515, score 0.8184, time 6.6 min, saved baseline_fold0_seed42.pt


Fold 1:  38%|███▊      | 19/50 [04:43<07:43, 14.94s/epoch, best=0.7871, score=0.7837, seconds=14]


Fold 1: best epoch 10, binary F1 0.9799, macro F1 0.5942, score 0.7871, time 4.7 min, saved baseline_fold1_seed42.pt


Fold 2:  56%|█████▌    | 28/50 [06:40<05:14, 14.31s/epoch, best=0.8234, score=0.8050, seconds=13]


Fold 2: best epoch 19, binary F1 0.9864, macro F1 0.6604, score 0.8234, time 6.7 min, saved baseline_fold2_seed42.pt


Fold 3:  42%|████▏     | 21/50 [05:06<07:03, 14.59s/epoch, best=0.8256, score=0.8103, seconds=14]


Fold 3: best epoch 12, binary F1 0.9897, macro F1 0.6614, score 0.8256, time 5.1 min, saved baseline_fold3_seed42.pt


Fold 4:  36%|███▌      | 18/50 [04:25<07:51, 14.74s/epoch, best=0.7857, score=0.7762, seconds=13]

Fold 4: best epoch 9, binary F1 0.9765, macro F1 0.5949, score 0.7857, time 4.4 min, saved baseline_fold4_seed42.pt


In [21]:
# The mean across folds is the main CV comparison; pooled OOF F1 is also shown.
print("Fold | Train | Val  | Best epoch | Binary F1 | Macro F1 | Score  | Minutes")
print("-" * 78)
for row in cv_results:
    print(
        f"{row['fold']:>4} | {row['train_n']:>5} | {row['val_n']:>4} | "
        f"{row['best_epoch']:>10} | {row['binary_f1']:.4f}    | "
        f"{row['macro_f1']:.4f}   | {row['score']:.4f} | {row['minutes']:.1f}"
    )

for metric in ("binary_f1", "macro_f1", "score"):
    values = np.array([row[metric] for row in cv_results])
    print(f"{metric}: mean {values.mean():.4f}, SD {values.std(ddof=1):.4f}")

pooled_metrics = competition_score(oof_true, oof_pred)
print(
    f"Pooled out-of-fold ({len(oof_true)} sequences): "
    f"binary F1 {pooled_metrics['binary_f1']:.4f}, "
    f"macro F1 {pooled_metrics['macro_f1']:.4f}, "
    f"score {pooled_metrics['score']:.4f}"
)
print(f"Total five-fold time: {sum(row['minutes'] for row in cv_results):.1f} minutes")
print("Best-epoch checkpoints:")
for row in cv_results:
    print(f"  Fold {row['fold']}: {row['checkpoint']}")


Fold | Train | Val  | Best epoch | Binary F1 | Macro F1 | Score  | Minutes
------------------------------------------------------------------------------
   0 |  6427 | 1627 |         20 | 0.9853    | 0.6515   | 0.8184 | 6.6
   1 |  6519 | 1535 |         10 | 0.9799    | 0.5942   | 0.7871 | 4.7
   2 |  6423 | 1631 |         19 | 0.9864    | 0.6604   | 0.8234 | 6.7
   3 |  6428 | 1626 |         12 | 0.9897    | 0.6614   | 0.8256 | 5.1
   4 |  6419 | 1635 |          9 | 0.9765    | 0.5949   | 0.7857 | 4.4
binary_f1: mean 0.9836, SD 0.0053
macro_f1: mean 0.6325, SD 0.0349
score: mean 0.8080, SD 0.0199
Pooled out-of-fold (8054 sequences): binary F1 0.9836, macro F1 0.6337, score 0.8087
Total five-fold time: 27.6 minutes
Best-epoch checkpoints:
  Fold 0: c:\Users\yuhen\Documents\GithubRepos\SC4000-Group-Assignment\model\baseline_fold0_seed42.pt
  Fold 1: c:\Users\yuhen\Documents\GithubRepos\SC4000-Group-Assignment\model\baseline_fold1_seed42.pt
  Fold 2: c:\Users\yuhen\Documents\GithubRepos

## Nine-class confusion matrix

Rows are the true classes and columns are the predicted classes. Gesture IDs 0–7 are the eight BFRB classes; all other labels are combined as **Non-BFRB**. The cells show counts and row percentages. This uses each fold's predictions from its best validation epoch.

In [22]:
# Display the competition's nine classes without adding a plotting dependency.
from html import escape
from IPython.display import HTML, display
from sklearn.metrics import confusion_matrix, f1_score

true_nine = np.where(oof_true < 8, oof_true, 8)
pred_nine = np.where(oof_pred < 8, oof_pred, 8)
class_names = [f"Gesture {i}" for i in range(8)] + ["Non-BFRB"]
cm = confusion_matrix(true_nine, pred_nine, labels=np.arange(9))
row_totals = cm.sum(axis=1)
row_pct = np.divide(
    cm, row_totals[:, None],
    out=np.zeros_like(cm, dtype=float),
    where=row_totals[:, None] != 0,
)

header = "<tr><th>True \\ Predicted</th>" + "".join(
    f"<th>{escape(name)}</th>" for name in class_names
) + "<th>Support</th></tr>"
rows = []
for i, name in enumerate(class_names):
    cells = []
    for j in range(9):
        pct = row_pct[i, j]
        alpha = 0.08 + 0.75 * pct
        fg = "#ffffff" if pct >= 0.55 else "#14213d"
        cells.append(
            f'<td style="background:rgba(37,99,235,{alpha:.3f});color:{fg};'
            f'text-align:center;padding:8px;min-width:66px">'
            f'<strong>{cm[i, j]}</strong><br><small>{pct:.1%}</small></td>'
        )
    rows.append(
        f"<tr><th>{escape(name)}</th>" + "".join(cells)
        + f"<td style='text-align:center'>{row_totals[i]}</td></tr>"
    )

table = (
    '<div style="overflow-x:auto"><table style="border-collapse:collapse;'
    'font-family:system-ui,Arial,sans-serif;font-size:13px">'
    + header + "".join(rows) + "</table></div>"
)
display(HTML(table))

print("Per-class F1 (nine-class task):")
for i, name in enumerate(class_names):
    class_f1 = f1_score(true_nine, pred_nine, labels=[i], average="macro", zero_division=0)
    print(f"  {name:>10}: F1 {class_f1:.4f}, support {row_totals[i]}")


True \ Predicted,Gesture 0,Gesture 1,Gesture 2,Gesture 3,Gesture 4,Gesture 5,Gesture 6,Gesture 7,Non-BFRB,Support
Gesture 0,50179.5%,254.0%,172.7%,162.5%,182.9%,50.8%,274.3%,121.9%,91.4%,630
Gesture 1,284.4%,36257.4%,182.9%,8914.1%,61.0%,50.8%,568.9%,609.5%,71.1%,631
Gesture 2,243.8%,325.1%,31950.6%,11418.1%,538.4%,6710.6%,10.2%,91.4%,111.7%,630
Gesture 3,284.4%,10015.8%,10116.0%,34554.6%,121.9%,152.4%,50.8%,152.4%,111.7%,632
Gesture 4,193.0%,60.9%,7912.5%,30.5%,41165.0%,9815.5%,00.0%,30.5%,132.1%,632
Gesture 5,111.7%,60.9%,8914.1%,142.2%,8914.1%,40263.6%,10.2%,81.3%,121.9%,632
Gesture 6,436.8%,8313.1%,20.3%,101.6%,10.2%,00.0%,31950.3%,17327.3%,30.5%,634
Gesture 7,304.7%,568.8%,40.6%,40.6%,10.2%,10.2%,19330.4%,34153.8%,40.6%,634
Non-BFRB,130.4%,200.7%,90.3%,110.4%,100.3%,100.3%,00.0%,230.8%,290396.8%,2999


Per-class F1 (nine-class task):
   Gesture 0: F1 0.7551, support 630
   Gesture 1: F1 0.5481, support 631
   Gesture 2: F1 0.5032, support 630
   Gesture 3: F1 0.5574, support 632
   Gesture 4: F1 0.6667, support 632
   Gesture 5: F1 0.6510, support 632
   Gesture 6: F1 0.5162, support 634
   Gesture 7: F1 0.5336, support 634
    Non-BFRB: F1 0.9722, support 2999


# Ablation 1: Treat the time-of-flight grids as tiny images and process them with a small 2D CNN.

Each sensor gives an 8×8 distance grid, so preserving that spatial layout lets the model recognize shapes like “a face filling the view."


Run the shared setup (cell 1), dataset (cell 2), and training helpers (cell 4) first. 
Then run the cells below. They use the same participant-separated folds, seed, 
hyperparameters, 18-class loss, and competition score as the baseline. 
The ToF model saves its own checkpoints and results.


In [26]:
# Ablation setup: keep the baseline variables and checkpoints intact.
from model.tof_cnn_gru import FullSensorToFCNNGRU
from tqdm.auto import tqdm

tof_model_kwargs = {"num_classes": 18, "gru_hidden_size": 128, "dropout": 0.2}
tof_checkpoint_dir = repo / "model"
tof_checkpoint_dir.mkdir(parents=True, exist_ok=True)

print("ToF spatial CNN + GRU ablation")
print(f"Fold count: 5; seed: {SEED}; batch size: {BATCH_SIZE}")
print(f"Trainable parameters: {sum(p.numel() for p in FullSensorToFCNNGRU(**tof_model_kwargs).parameters()):,}")


ToF spatial CNN + GRU ablation
Fold count: 5; seed: 42; batch size: 16
Trainable parameters: 471,282


In [27]:
# Train the spatial ToF model independently on the same five folds.
tof_cv_results = []
tof_cv_history = []
tof_oof_true_parts, tof_oof_pred_parts = [], []

for tof_fold in range(5):
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    tof_fold_dir = repo / "data/ready_full" / f"fold{tof_fold}"
    tof_train = GestureDataset(tof_fold_dir, "train")
    tof_val = GestureDataset(tof_fold_dir, "val")
    tof_train_loader = DataLoader(
        tof_train, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=True,
        generator=torch.Generator().manual_seed(SEED),
    )
    tof_val_loader = DataLoader(
        tof_val, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True,
    )

    tof_model = FullSensorToFCNNGRU(**tof_model_kwargs).to(device)
    tof_criterion = nn.CrossEntropyLoss()
    tof_optimizer = torch.optim.AdamW(
        tof_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
    )
    tof_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        tof_optimizer, T_max=MAX_EPOCHS, eta_min=MIN_LEARNING_RATE,
    )
    tof_checkpoint_path = tof_checkpoint_dir / f"tof_cnn_gru_fold{tof_fold}_seed{SEED}.pt"

    tof_best_score = float("-inf")
    tof_best_epoch = 0
    tof_best_metrics = None
    tof_best_true = tof_best_pred = None
    tof_stale_epochs = 0

    torch.cuda.synchronize()
    tof_fold_start = time.perf_counter()
    tof_epoch_bar = tqdm(range(1, MAX_EPOCHS + 1), desc=f"ToF fold {tof_fold}", unit="epoch")
    for tof_epoch in tof_epoch_bar:
        tof_epoch_start = time.perf_counter()
        tof_train_loss = train_one_epoch(
            tof_model, tof_train_loader, tof_optimizer, tof_criterion,
            device, GRAD_CLIP_NORM,
        )
        tof_val_loss, tof_metrics, tof_y_true, tof_y_pred = validate(
            tof_model, tof_val_loader, tof_criterion, device,
        )
        tof_scheduler.step()
        torch.cuda.synchronize()
        tof_epoch_seconds = time.perf_counter() - tof_epoch_start

        tof_cv_history.append({
            "fold": tof_fold, "epoch": tof_epoch, "train_loss": tof_train_loss,
            "val_loss": tof_val_loss, **tof_metrics, "seconds": tof_epoch_seconds,
        })
        if tof_metrics["score"] > tof_best_score:
            tof_best_score = tof_metrics["score"]
            tof_best_epoch = tof_epoch
            tof_best_metrics = tof_metrics.copy()
            tof_best_true, tof_best_pred = tof_y_true.copy(), tof_y_pred.copy()
            tof_stale_epochs = 0
            tof_checkpoint = {
                "model_class": "model.tof_cnn_gru.FullSensorToFCNNGRU",
                "model_kwargs": tof_model_kwargs.copy(),
                "model_state_dict": {
                    name: tensor.detach().cpu().clone()
                    for name, tensor in tof_model.state_dict().items()
                },
                "fold": tof_fold,
                "seed": SEED,
                "best_epoch": tof_epoch,
                "val_metrics": {name: float(value) for name, value in tof_metrics.items()},
                "preprocessing_stats": f"data/ready_full/stats_fold{tof_fold}.npz",
            }
            tof_temporary_path = tof_checkpoint_path.with_suffix(".tmp")
            torch.save(tof_checkpoint, tof_temporary_path)
            tof_temporary_path.replace(tof_checkpoint_path)
        else:
            tof_stale_epochs += 1

        tof_epoch_bar.set_postfix(
            score=f"{tof_metrics['score']:.4f}",
            best=f"{tof_best_score:.4f}",
            seconds=f"{tof_epoch_seconds:.0f}",
        )
        if tof_stale_epochs >= PATIENCE:
            break

    torch.cuda.synchronize()
    tof_fold_seconds = time.perf_counter() - tof_fold_start
    tof_cv_results.append({
        "fold": tof_fold, "train_n": len(tof_train), "val_n": len(tof_val),
        "best_epoch": tof_best_epoch, **tof_best_metrics,
        "minutes": tof_fold_seconds / 60, "checkpoint": str(tof_checkpoint_path),
    })
    tof_oof_true_parts.append(tof_best_true)
    tof_oof_pred_parts.append(tof_best_pred)
    print(
        f"ToF fold {tof_fold}: best epoch {tof_best_epoch}, "
        f"binary F1 {tof_best_metrics['binary_f1']:.4f}, "
        f"macro F1 {tof_best_metrics['macro_f1']:.4f}, "
        f"score {tof_best_score:.4f}, time {tof_fold_seconds / 60:.1f} min, "
        f"saved {tof_checkpoint_path.name}"
    )

tof_oof_true = np.concatenate(tof_oof_true_parts)
tof_oof_pred = np.concatenate(tof_oof_pred_parts)


ToF fold 0:  40%|████      | 20/50 [07:18<10:57, 21.91s/epoch, best=0.8273, score=0.8213, seconds=21]


ToF fold 0: best epoch 11, binary F1 0.9887, macro F1 0.6658, score 0.8273, time 7.3 min, saved tof_cnn_gru_fold0_seed42.pt


ToF fold 1:  54%|█████▍    | 27/50 [09:40<08:14, 21.49s/epoch, best=0.8005, score=0.7913, seconds=22]


ToF fold 1: best epoch 18, binary F1 0.9835, macro F1 0.6176, score 0.8005, time 9.7 min, saved tof_cnn_gru_fold1_seed42.pt


ToF fold 2:  54%|█████▍    | 27/50 [12:33<10:41, 27.90s/epoch, best=0.8299, score=0.8212, seconds=19]


ToF fold 2: best epoch 18, binary F1 0.9903, macro F1 0.6696, score 0.8299, time 12.6 min, saved tof_cnn_gru_fold2_seed42.pt


ToF fold 3:  44%|████▍     | 22/50 [07:26<09:28, 20.31s/epoch, best=0.8302, score=0.8166, seconds=20]


ToF fold 3: best epoch 13, binary F1 0.9911, macro F1 0.6692, score 0.8302, time 7.4 min, saved tof_cnn_gru_fold3_seed42.pt


ToF fold 4:  48%|████▊     | 24/50 [08:38<09:21, 21.59s/epoch, best=0.8011, score=0.7986, seconds=20]

ToF fold 4: best epoch 15, binary F1 0.9799, macro F1 0.6222, score 0.8011, time 8.6 min, saved tof_cnn_gru_fold4_seed42.pt


In [28]:
# Report the ToF ablation and compare mean fold scores with the baseline.
print("Fold | Train | Val  | Best epoch | Binary F1 | Macro F1 | Score  | Minutes")
print("-" * 78)
for tof_row in tof_cv_results:
    print(
        f"{tof_row['fold']:>4} | {tof_row['train_n']:>5} | {tof_row['val_n']:>4} | "
        f"{tof_row['best_epoch']:>10} | {tof_row['binary_f1']:.4f}    | "
        f"{tof_row['macro_f1']:.4f}   | {tof_row['score']:.4f} | {tof_row['minutes']:.1f}"
    )

for tof_metric in ("binary_f1", "macro_f1", "score"):
    tof_values = np.array([row[tof_metric] for row in tof_cv_results])
    print(f"ToF {tof_metric}: mean {tof_values.mean():.4f}, SD {tof_values.std(ddof=1):.4f}")

tof_pooled_metrics = competition_score(tof_oof_true, tof_oof_pred)
print(
    f"Pooled ToF out-of-fold ({len(tof_oof_true)} sequences): "
    f"binary F1 {tof_pooled_metrics['binary_f1']:.4f}, "
    f"macro F1 {tof_pooled_metrics['macro_f1']:.4f}, "
    f"score {tof_pooled_metrics['score']:.4f}"
)
print(f"Total five-fold time: {sum(row['minutes'] for row in tof_cv_results):.1f} minutes")

if "cv_results" in globals() and len(cv_results) == len(tof_cv_results):
    print("\nAblation: mean five-fold scores (ToF CNN minus baseline)")
    print("Metric       | Baseline | ToF CNN | Delta")
    print("-" * 45)
    for tof_metric in ("binary_f1", "macro_f1", "score"):
        baseline_mean = np.mean([row[tof_metric] for row in cv_results])
        tof_mean = np.mean([row[tof_metric] for row in tof_cv_results])
        print(f"{tof_metric:<12} | {baseline_mean:.4f}   | {tof_mean:.4f}  | {tof_mean - baseline_mean:+.4f}")
else:
    print("Run the baseline five-fold cells to display the side-by-side comparison.")


Fold | Train | Val  | Best epoch | Binary F1 | Macro F1 | Score  | Minutes
------------------------------------------------------------------------------
   0 |  6427 | 1627 |         11 | 0.9887    | 0.6658   | 0.8273 | 7.3
   1 |  6519 | 1535 |         18 | 0.9835    | 0.6176   | 0.8005 | 9.7
   2 |  6423 | 1631 |         18 | 0.9903    | 0.6696   | 0.8299 | 12.6
   3 |  6428 | 1626 |         13 | 0.9911    | 0.6692   | 0.8302 | 7.4
   4 |  6419 | 1635 |         15 | 0.9799    | 0.6222   | 0.8011 | 8.6
ToF binary_f1: mean 0.9867, SD 0.0048
ToF macro_f1: mean 0.6489, SD 0.0265
ToF score: mean 0.8178, SD 0.0156
Pooled ToF out-of-fold (8054 sequences): binary F1 0.9867, macro F1 0.6507, score 0.8187
Total five-fold time: 45.6 minutes

Ablation: mean five-fold scores (ToF CNN minus baseline)
Metric       | Baseline | ToF CNN | Delta
---------------------------------------------
binary_f1    | 0.9836   | 0.9867  | +0.0031
macro_f1     | 0.6325   | 0.6489  | +0.0164
score        | 0.8080  

# Ablation 2: Give the model two output heads that mirror the scoring metric.
One head decides BFRB vs. not, and the other picks which BFRB. You’re scored on exactly those two questions, so optimizing them directly, and tuning the decision threshold, raises your score.

Run the initial setup and `GestureDataset` cells above first. This section uses the 
same participant-separated folds, seed, optimizer settings, and early stopping as 
Ablation 1. It trains the binary and eight-gesture heads together, scores at a fixed 
0.5 binary threshold, and writes separate checkpoints. The threshold is **not** 
selected on these validation folds.


In [29]:
# Two-head setup: keep the earlier model, results, and checkpoints intact.
from functools import partial
from model.tof_cnn_gru_two_head import FullSensorToFTwoHeadCNNGRU
from utils.train_two_head import two_head_loss, train_two_head_one_epoch, validate_two_head
from tqdm.auto import tqdm

TWO_HEAD_GESTURE_WEIGHT = 1.0
TWO_HEAD_THRESHOLD = 0.5
two_head_criterion = partial(two_head_loss, gesture_weight=TWO_HEAD_GESTURE_WEIGHT)
two_head_model_kwargs = {"gru_hidden_size": 128, "dropout": 0.2}
two_head_checkpoint_dir = repo / "model"
two_head_checkpoint_dir.mkdir(parents=True, exist_ok=True)

print("Two-head ToF CNN + GRU ablation")
print(f"Fold count: 5; seed: {SEED}; batch size: {BATCH_SIZE}")
print(f"Gesture loss weight: {TWO_HEAD_GESTURE_WEIGHT}; decision threshold: {TWO_HEAD_THRESHOLD}")
print(f"Trainable parameters: {sum(p.numel() for p in FullSensorToFTwoHeadCNNGRU(**two_head_model_kwargs).parameters()):,}")


Two-head ToF CNN + GRU ablation
Fold count: 5; seed: 42; batch size: 16
Gesture loss weight: 1.0; decision threshold: 0.5
Trainable parameters: 468,969


In [30]:
# Train the two-head model independently on the same five folds.
two_head_cv_results = []
two_head_cv_history = []
two_head_oof_true_parts, two_head_oof_pred_parts = [], []
two_head_oof_binary_parts, two_head_oof_gesture_parts = [], []
two_head_oof_fold_parts = []

for two_head_fold in range(5):
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    two_head_fold_dir = repo / "data/ready_full" / f"fold{two_head_fold}"
    two_head_train = GestureDataset(two_head_fold_dir, "train")
    two_head_val = GestureDataset(two_head_fold_dir, "val")
    two_head_train_loader = DataLoader(
        two_head_train, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=True,
        generator=torch.Generator().manual_seed(SEED),
    )
    two_head_val_loader = DataLoader(
        two_head_val, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True,
    )

    two_head_model = FullSensorToFTwoHeadCNNGRU(**two_head_model_kwargs).to(device)
    two_head_optimizer = torch.optim.AdamW(
        two_head_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
    )
    two_head_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        two_head_optimizer, T_max=MAX_EPOCHS, eta_min=MIN_LEARNING_RATE,
    )
    two_head_checkpoint_path = (
        two_head_checkpoint_dir / f"tof_two_head_fold{two_head_fold}_seed{SEED}.pt"
    )

    two_head_best_score = float("-inf")
    two_head_best_epoch = 0
    two_head_best_metrics = None
    two_head_best_true = two_head_best_pred = None
    two_head_best_binary = two_head_best_gesture = None
    two_head_stale_epochs = 0

    torch.cuda.synchronize()
    two_head_fold_start = time.perf_counter()
    two_head_epoch_bar = tqdm(
        range(1, MAX_EPOCHS + 1), desc=f"Two-head fold {two_head_fold}", unit="epoch"
    )
    for two_head_epoch in two_head_epoch_bar:
        two_head_epoch_start = time.perf_counter()
        two_head_train_loss = train_two_head_one_epoch(
            two_head_model, two_head_train_loader, two_head_optimizer,
            two_head_criterion, device, GRAD_CLIP_NORM,
        )
        (
            two_head_val_loss, two_head_metrics, two_head_y_true, two_head_y_pred,
            two_head_binary_logits, two_head_gesture_logits,
        ) = validate_two_head(
            two_head_model, two_head_val_loader, two_head_criterion, device,
            threshold=TWO_HEAD_THRESHOLD,
        )
        two_head_scheduler.step()
        torch.cuda.synchronize()
        two_head_epoch_seconds = time.perf_counter() - two_head_epoch_start

        two_head_cv_history.append({
            "fold": two_head_fold, "epoch": two_head_epoch,
            "train_loss": two_head_train_loss, "val_loss": two_head_val_loss,
            **two_head_metrics, "seconds": two_head_epoch_seconds,
        })
        if two_head_metrics["score"] > two_head_best_score:
            two_head_best_score = two_head_metrics["score"]
            two_head_best_epoch = two_head_epoch
            two_head_best_metrics = two_head_metrics.copy()
            two_head_best_true = two_head_y_true.copy()
            two_head_best_pred = two_head_y_pred.copy()
            two_head_best_binary = two_head_binary_logits.clone()
            two_head_best_gesture = two_head_gesture_logits.clone()
            two_head_stale_epochs = 0
            two_head_checkpoint = {
                "model_class": "model.tof_cnn_gru_two_head.FullSensorToFTwoHeadCNNGRU",
                "model_kwargs": two_head_model_kwargs.copy(),
                "model_state_dict": {
                    name: tensor.detach().cpu().clone()
                    for name, tensor in two_head_model.state_dict().items()
                },
                "fold": two_head_fold,
                "seed": SEED,
                "best_epoch": two_head_epoch,
                "val_metrics": {
                    name: float(value) for name, value in two_head_metrics.items()
                },
                "gesture_weight": TWO_HEAD_GESTURE_WEIGHT,
                "threshold": TWO_HEAD_THRESHOLD,
                "preprocessing_stats": f"data/ready_full/stats_fold{two_head_fold}.npz",
            }
            two_head_temporary_path = two_head_checkpoint_path.with_suffix(".tmp")
            torch.save(two_head_checkpoint, two_head_temporary_path)
            two_head_temporary_path.replace(two_head_checkpoint_path)
        else:
            two_head_stale_epochs += 1

        two_head_epoch_bar.set_postfix(
            score=f"{two_head_metrics['score']:.4f}",
            best=f"{two_head_best_score:.4f}",
            seconds=f"{two_head_epoch_seconds:.0f}",
        )
        if two_head_stale_epochs >= PATIENCE:
            break

    torch.cuda.synchronize()
    two_head_fold_seconds = time.perf_counter() - two_head_fold_start
    two_head_cv_results.append({
        "fold": two_head_fold, "train_n": len(two_head_train),
        "val_n": len(two_head_val), "best_epoch": two_head_best_epoch,
        **two_head_best_metrics, "minutes": two_head_fold_seconds / 60,
        "checkpoint": str(two_head_checkpoint_path),
    })
    two_head_oof_true_parts.append(two_head_best_true)
    two_head_oof_pred_parts.append(two_head_best_pred)
    two_head_oof_binary_parts.append(two_head_best_binary)
    two_head_oof_gesture_parts.append(two_head_best_gesture)
    two_head_oof_fold_parts.append(
        np.full(len(two_head_best_true), two_head_fold, dtype=np.int8)
    )
    print(
        f"Two-head fold {two_head_fold}: best epoch {two_head_best_epoch}, "
        f"binary F1 {two_head_best_metrics['binary_f1']:.4f}, "
        f"macro F1 {two_head_best_metrics['macro_f1']:.4f}, "
        f"score {two_head_best_score:.4f}, time {two_head_fold_seconds / 60:.1f} min, "
        f"saved {two_head_checkpoint_path.name}"
    )

two_head_oof_true = np.concatenate(two_head_oof_true_parts)
two_head_oof_pred = np.concatenate(two_head_oof_pred_parts)
two_head_oof_binary_logits = torch.cat(two_head_oof_binary_parts)
two_head_oof_gesture_logits = torch.cat(two_head_oof_gesture_parts)
two_head_oof_fold = np.concatenate(two_head_oof_fold_parts)


Two-head fold 0:  40%|████      | 20/50 [11:06<16:40, 33.34s/epoch, best=0.8168, score=0.8162, seconds=34]


Two-head fold 0: best epoch 11, binary F1 0.9814, macro F1 0.6521, score 0.8168, time 11.1 min, saved tof_two_head_fold0_seed42.pt


Two-head fold 1:  46%|████▌     | 23/50 [12:00<14:05, 31.31s/epoch, best=0.7924, score=0.7890, seconds=27]


Two-head fold 1: best epoch 14, binary F1 0.9839, macro F1 0.6009, score 0.7924, time 12.0 min, saved tof_two_head_fold1_seed42.pt


Two-head fold 2:  44%|████▍     | 22/50 [11:30<14:38, 31.38s/epoch, best=0.8181, score=0.8070, seconds=29]


Two-head fold 2: best epoch 13, binary F1 0.9859, macro F1 0.6502, score 0.8181, time 11.5 min, saved tof_two_head_fold2_seed42.pt


Two-head fold 3:  44%|████▍     | 22/50 [11:28<14:35, 31.28s/epoch, best=0.8272, score=0.8172, seconds=30]


Two-head fold 3: best epoch 13, binary F1 0.9867, macro F1 0.6676, score 0.8272, time 11.5 min, saved tof_two_head_fold3_seed42.pt


Two-head fold 4:  76%|███████▌  | 38/50 [19:08<06:02, 30.22s/epoch, best=0.7970, score=0.7874, seconds=31]

Two-head fold 4: best epoch 29, binary F1 0.9693, macro F1 0.6247, score 0.7970, time 19.1 min, saved tof_two_head_fold4_seed42.pt


In [31]:
# Report two-head CV and compare with the single-head ToF CNN.
print("Fold | Train | Val  | Best epoch | Binary F1 | Macro F1 | Score  | Minutes")
print("-" * 78)
for two_head_row in two_head_cv_results:
    print(
        f"{two_head_row['fold']:>4} | {two_head_row['train_n']:>5} | "
        f"{two_head_row['val_n']:>4} | {two_head_row['best_epoch']:>10} | "
        f"{two_head_row['binary_f1']:.4f}    | "
        f"{two_head_row['macro_f1']:.4f}   | "
        f"{two_head_row['score']:.4f} | {two_head_row['minutes']:.1f}"
    )

for two_head_metric in ("binary_f1", "macro_f1", "score"):
    two_head_values = np.array([row[two_head_metric] for row in two_head_cv_results])
    print(
        f"Two-head {two_head_metric}: mean {two_head_values.mean():.4f}, "
        f"SD {two_head_values.std(ddof=1):.4f}"
    )

two_head_pooled_metrics = competition_score(two_head_oof_true, two_head_oof_pred)
print(
    f"Pooled two-head out-of-fold ({len(two_head_oof_true)} sequences): "
    f"binary F1 {two_head_pooled_metrics['binary_f1']:.4f}, "
    f"macro F1 {two_head_pooled_metrics['macro_f1']:.4f}, "
    f"score {two_head_pooled_metrics['score']:.4f}"
)
print(f"Total five-fold time: {sum(row['minutes'] for row in two_head_cv_results):.1f} minutes")

if "tof_cv_results" in globals() and len(tof_cv_results) == len(two_head_cv_results):
    print("\nAblation: mean five-fold scores (two heads minus single head)")
    print("Metric       | ToF CNN | Two heads | Delta")
    print("-" * 47)
    for two_head_metric in ("binary_f1", "macro_f1", "score"):
        tof_mean = np.mean([row[two_head_metric] for row in tof_cv_results])
        two_head_mean = np.mean([row[two_head_metric] for row in two_head_cv_results])
        print(
            f"{two_head_metric:<12} | {tof_mean:.4f}  | "
            f"{two_head_mean:.4f}    | {two_head_mean - tof_mean:+.4f}"
        )
else:
    print("Run the single-head ToF five-fold cells to display the side-by-side comparison.")


Fold | Train | Val  | Best epoch | Binary F1 | Macro F1 | Score  | Minutes
------------------------------------------------------------------------------
   0 |  6427 | 1627 |         11 | 0.9814    | 0.6521   | 0.8168 | 11.1
   1 |  6519 | 1535 |         14 | 0.9839    | 0.6009   | 0.7924 | 12.0
   2 |  6423 | 1631 |         13 | 0.9859    | 0.6502   | 0.8181 | 11.5
   3 |  6428 | 1626 |         13 | 0.9867    | 0.6676   | 0.8272 | 11.5
   4 |  6419 | 1635 |         29 | 0.9693    | 0.6247   | 0.7970 | 19.1
Two-head binary_f1: mean 0.9814, SD 0.0071
Two-head macro_f1: mean 0.6391, SD 0.0263
Two-head score: mean 0.8103, SD 0.0149
Pooled two-head out-of-fold (8054 sequences): binary F1 0.9815, macro F1 0.6421, score 0.8118
Total five-fold time: 65.2 minutes

Ablation: mean five-fold scores (two heads minus single head)
Metric       | ToF CNN | Two heads | Delta
-----------------------------------------------
binary_f1    | 0.9867  | 0.9814    | -0.0053
macro_f1     | 0.6489  | 0.6391   

In [32]:
# Pointer 6 stability check: same five participant folds, three training seeds.
# Run the setup/hyperparameter cell and GestureDataset cell first in a fresh kernel.
# Reuse the completed seed-42 checkpoints; train seeds 43 and 44 without changing the model or hyperparameters.
from model.tof_cnn_gru import FullSensorToFCNNGRU
from utils.train import train_one_epoch, validate
from tqdm.auto import tqdm

stability_seeds = (42, 43, 44)
stability_model_kwargs = {"num_classes": 18, "gru_hidden_size": 128, "dropout": 0.2}
stability_results = []
stability_checkpoint_dir = repo / "model"

# Check all reused checkpoints before starting the two long training runs.
for fold in range(5):
    path = stability_checkpoint_dir / f"tof_cnn_gru_fold{fold}_seed42.pt"
    if not path.is_file():
        raise FileNotFoundError(f"Missing original pointer-6 checkpoint: {path}")

for seed in stability_seeds:
    for fold in range(5):
        checkpoint_path = stability_checkpoint_dir / f"tof_cnn_gru_fold{fold}_seed{seed}.pt"
        if seed == 42:
            checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
            if (checkpoint["model_class"] != "model.tof_cnn_gru.FullSensorToFCNNGRU"
                    or checkpoint["model_kwargs"] != stability_model_kwargs
                    or checkpoint["fold"] != fold or checkpoint["seed"] != seed):
                raise ValueError(f"Unexpected model, fold, or seed in {checkpoint_path}")
            best_epoch = checkpoint["best_epoch"]
            best_metrics = checkpoint["val_metrics"]
            elapsed_minutes = None
            del checkpoint
        else:
            random.seed(seed)
            np.random.seed(seed)
            torch.manual_seed(seed)
            torch.cuda.manual_seed_all(seed)

            fold_dir = repo / "data/ready_full" / f"fold{fold}"
            train_data = GestureDataset(fold_dir, "train")
            val_data = GestureDataset(fold_dir, "val")
            train_loader = DataLoader(
                train_data, batch_size=BATCH_SIZE, shuffle=True,
                num_workers=NUM_WORKERS, pin_memory=True,
                generator=torch.Generator().manual_seed(seed),
            )
            val_loader = DataLoader(
                val_data, batch_size=BATCH_SIZE, shuffle=False,
                num_workers=NUM_WORKERS, pin_memory=True,
            )
            model = FullSensorToFCNNGRU(**stability_model_kwargs).to(device)
            criterion = nn.CrossEntropyLoss()
            optimizer = torch.optim.AdamW(
                model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
            )
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=MAX_EPOCHS, eta_min=MIN_LEARNING_RATE,
            )
            best_score = float("-inf")
            best_epoch = 0
            best_metrics = None
            stale_epochs = 0
            torch.cuda.synchronize()
            fold_start = time.perf_counter()

            for epoch in tqdm(range(1, MAX_EPOCHS + 1), desc=f"ToF seed {seed} fold {fold}", unit="epoch"):
                train_one_epoch(model, train_loader, optimizer, criterion, device, GRAD_CLIP_NORM)
                _, metrics, _, _ = validate(model, val_loader, criterion, device)
                scheduler.step()
                if metrics["score"] > best_score:
                    best_score = metrics["score"]
                    best_epoch = epoch
                    best_metrics = metrics.copy()
                    stale_epochs = 0
                    checkpoint = {
                        "model_class": "model.tof_cnn_gru.FullSensorToFCNNGRU",
                        "model_kwargs": stability_model_kwargs.copy(),
                        "model_state_dict": {
                            name: tensor.detach().cpu().clone()
                            for name, tensor in model.state_dict().items()
                        },
                        "fold": fold, "seed": seed, "best_epoch": epoch,
                        "val_metrics": {name: float(value) for name, value in metrics.items()},
                        "preprocessing_stats": f"data/ready_full/stats_fold{fold}.npz",
                    }
                    temporary_path = checkpoint_path.with_suffix(".tmp")
                    torch.save(checkpoint, temporary_path)
                    temporary_path.replace(checkpoint_path)
                else:
                    stale_epochs += 1
                if stale_epochs >= PATIENCE:
                    break

            torch.cuda.synchronize()
            elapsed_minutes = (time.perf_counter() - fold_start) / 60
            del model, train_loader, val_loader, train_data, val_data

        stability_results.append({
            "seed": seed, "fold": fold, "best_epoch": best_epoch,
            **best_metrics, "minutes": elapsed_minutes,
            "checkpoint": str(checkpoint_path),
        })
        print(
            f"Seed {seed}, fold {fold}: epoch {best_epoch}, "
            f"binary F1 {best_metrics['binary_f1']:.4f}, "
            f"macro F1 {best_metrics['macro_f1']:.4f}, "
            f"score {best_metrics['score']:.4f}"
            + (" (saved run)" if seed == 42 else f", {elapsed_minutes:.1f} min")
        )

print("\nPointer 6: five-fold mean by training seed")
print("Seed | Binary F1 | Macro F1 | Score  | Fold-score SD")
print("-" * 57)
seed_means = []
for seed in stability_seeds:
    rows = [row for row in stability_results if row["seed"] == seed]
    scores = np.array([row["score"] for row in rows])
    seed_means.append(scores.mean())
    print(
        f"{seed:>4} | {np.mean([row['binary_f1'] for row in rows]):.4f}    | "
        f"{np.mean([row['macro_f1'] for row in rows]):.4f}   | "
        f"{scores.mean():.4f} | {scores.std(ddof=1):.4f}"
    )
seed_means = np.array(seed_means)
print(
    f"Across three seed means: {seed_means.mean():.4f} ± {seed_means.std(ddof=1):.4f} SD; "
    f"range {seed_means.min():.4f}–{seed_means.max():.4f}"
)
print(f"Difference from original seed-42 mean: {seed_means.mean() - seed_means[0]:+.4f}")
print("The SD above measures seed variation; the same participant folds are reused for all seeds.")


Seed 42, fold 0: epoch 11, binary F1 0.9887, macro F1 0.6658, score 0.8273 (saved run)
Seed 42, fold 1: epoch 18, binary F1 0.9835, macro F1 0.6176, score 0.8005 (saved run)
Seed 42, fold 2: epoch 18, binary F1 0.9903, macro F1 0.6696, score 0.8299 (saved run)
Seed 42, fold 3: epoch 13, binary F1 0.9911, macro F1 0.6692, score 0.8302 (saved run)
Seed 42, fold 4: epoch 15, binary F1 0.9799, macro F1 0.6222, score 0.8011 (saved run)


ToF seed 43 fold 0:  36%|███▌      | 18/50 [05:22<09:34, 17.94s/epoch]


Seed 43, fold 0: epoch 9, binary F1 0.9857, macro F1 0.6622, score 0.8240, 5.4 min


ToF seed 43 fold 1: 100%|██████████| 50/50 [13:22<00:00, 16.04s/epoch]


Seed 43, fold 1: epoch 43, binary F1 0.9856, macro F1 0.6036, score 0.7946, 13.4 min


...

Seed 44, fold 2: epoch 22, binary F1 0.9932, macro F1 0.6725, score 0.8328, 8.7 min


ToF seed 44 fold 3:  74%|███████▍  | 37/50 [10:17<03:37, 16.69s/epoch]


Seed 44, fold 3: epoch 28, binary F1 0.9881, macro F1 0.6801, score 0.8341, 10.3 min


ToF seed 44 fold 4:  68%|██████▊   | 34/50 [09:33<04:30, 16.88s/epoch]


Seed 44, fold 4: epoch 25, binary F1 0.9795, macro F1 0.6284, score 0.8039, 9.6 min

Pointer 6: five-fold mean by training seed
Seed | Binary F1 | Macro F1 | Score  | Fold-score SD
---------------------------------------------------------
  42 | 0.9867    | 0.6489   | 0.8178 | 0.0156
  43 | 0.9860    | 0.6489   | 0.8175 | 0.0178
  44 | 0.9866    | 0.6519   | 0.8192 | 0.0177
Across three seed means: 0.8182 ± 0.0009 SD; range 0.8175–0.8192
Difference from original seed-42 mean: +0.0004
The SD above measures seed variation; the same participant folds are reused for all seeds.
